# Prototype get gfw sar data

In [1]:
# gfw_s2_vessel_detections_min.py
from google.cloud import bigquery
import pandas as pd
import os 
import gfwapiclient as gfw

In [2]:
year = 2024
temporal_granularity = 'DAILY'

start_date= f"{str(year)}-01-01"
end_date= f"{str(year)}-01-31" 
spatial_resolution= "HIGH" # LOW or MEDIUM or HIGH

product_name = "public-global-fishing-effort:latest"

lat_name = 'lat'
lon_name = 'lon'
lat_min = 45.4698
lat_max = 45.8078
lon_min = 13.141159647873394
lon_max = 13.8238

In [3]:
from google.cloud import bigquery
from google.oauth2 import service_account

KEY_PATH = "/home/samuele/Desktop/pat/double-catfish-475813-h4-689d25711937.json"
creds = service_account.Credentials.from_service_account_file(KEY_PATH)

# Legge direttamente dal file il project_id corretto
PROJECT_ID = creds.project_id
print("Project ID:", PROJECT_ID)

client = bigquery.Client(project=PROJECT_ID, credentials=creds, location="US")

for row in client.query("SELECT 1 AS test").result():
    print(row.test)


Project ID: double-catfish-475813-h4
1


In [4]:
query = """
SELECT *
FROM `global-fishing-watch.pipe_sentinel2_v1_published.detect_scene_match_pipe_v3`
LIMIT 5
"""
for row in client.query(query).result():
    print(row)


Row(('S2B_MSIL1C_20250824T071619_N0511_R006_T38RQT_20250824T091351;48.12223200;29.5378560', 'S2B_MSIL1C_20250824T071619_N0511_R006_T38RQT_20250824T091351', '447004712', 'AIS', '447004712-2025-08-24T04:49:54.000000Z-1', datetime.datetime(2025, 8, 24, 7, 31, 44, 884000, tzinfo=datetime.timezone.utc), 29.537856, 48.122232, 5.02, 196.04281826366304, 23.799999999999997, 0.927, 0.0, 0.024220547880492774, None, 29.541508, 29.53465, 48.115572, 48.116475, 101.8, 226.1, 16.1, 23.7, datetime.datetime(2025, 8, 24, 7, 22, 53, tzinfo=datetime.timezone.utc), datetime.datetime(2025, 8, 24, 7, 34, 48, tzinfo=datetime.timezone.utc), 14.5408143249, 0.2178775817, 0.016224528569728135, False, None, False, datetime.date(2025, 8, 24)), {'detect_id': 0, 'scene_id': 1, 'ssvid': 2, 'source': 3, 'seg_id': 4, 'detect_timestamp': 5, 'detect_lat': 6, 'detect_lon': 7, 'speed_kn_inferred': 8, 'heading_deg_inferred': 9, 'length_m_inferred': 10, 'presence_score': 11, 'matching_score': 12, 'matching_score_secondary': 13

In [5]:
table = client.get_table("global-fishing-watch.pipe_sentinel2_v1_published.detect_scene_match_pipe_v3")
for col in table.schema:
    print(f'{col.name},')


detect_id,
scene_id,
ssvid,
source,
seg_id,
detect_timestamp,
detect_lat,
detect_lon,
speed_kn_inferred,
heading_deg_inferred,
length_m_inferred,
presence_score,
matching_score,
matching_score_secondary,
matching_confidence,
ais_lat_before,
ais_lat_after,
ais_lon_before,
ais_lon_after,
ais_course_before,
ais_course_after,
ais_speed_kn_before,
ais_speed_kn_after,
ais_timestamp_before,
ais_timestamp_after,
best_ais_length_m,
other_score_nn,
cloud_score,
likely_infrastructure,
structure_id,
potential_ice,
date,


In [ ]:
# --- Librerie ---
from google.cloud import bigquery
from google.oauth2 import service_account
import pandas as pd

# --- Credenziali e connessione ---
KEY_PATH = "/home/samuele/Desktop/pat/double-catfish-475813-h4-689d25711937.json"

creds = service_account.Credentials.from_service_account_file(
    KEY_PATH,
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)

PROJECT_ID = creds.project_id
client = bigquery.Client(project=PROJECT_ID, credentials=creds, location="US")

# --- Parametri ---
DATE_START = f"{start_date}T00:00:00"
DATE_END   = f"{end_date}T23:59:59"
OUT_PATH   = f"../data/raw/s2_vessel_detections_{start_date}.parquet"

# Bounding box (lon_min, lat_min, lon_max, lat_max)
USE_BBOX = True

# --- Query ---
# --- Query aggiornata ---
base_sql = """
SELECT
  detect_id,
  detect_timestamp,
  detect_lat,
  detect_lon,
  speed_kn_inferred,
  heading_deg_inferred,
  length_m_inferred,
  presence_score,
  matching_score,
  cloud_score,
  likely_infrastructure,
  structure_id,
  date
  FROM `global-fishing-watch.pipe_sentinel2_v1_published.detect_scene_match_pipe_v3`
WHERE detect_timestamp BETWEEN @start AND @end
"""

if USE_BBOX:
    base_sql += """
      AND detect_lon BETWEEN @lon_min AND @lon_max
      AND detect_lat BETWEEN @lat_min AND @lat_max
    """


# --- Parametri della query ---
params = [
    bigquery.ScalarQueryParameter("start", "TIMESTAMP", DATE_START),
    bigquery.ScalarQueryParameter("end", "TIMESTAMP", DATE_END),
]

if USE_BBOX:
    params += [
        bigquery.ScalarQueryParameter("lon_min", "FLOAT64", lon_min),
        bigquery.ScalarQueryParameter("lon_max", "FLOAT64", lon_max),
        bigquery.ScalarQueryParameter("lat_min", "FLOAT64", lat_min),
        bigquery.ScalarQueryParameter("lat_max", "FLOAT64", lat_max),
    ]

job_config = bigquery.QueryJobConfig(query_parameters=params)

# --- Esecuzione ---
print("⏳ Query in esecuzione su BigQuery...")
job = client.query(base_sql, job_config=job_config)
df = job.result().to_dataframe(create_bqstorage_client=True)

print(f"✅ Dati scaricati: {len(df)} righe")

# --- Salvataggio ---
df.to_parquet(OUT_PATH, index=False)
print(f"💾 Salvato in {OUT_PATH}")


⏳ Query in esecuzione su BigQuery...


/home/samuele/miniconda3/envs/nemoai/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


✅ Dati scaricati: 352 righe
💾 Salvato in ../data/raw/s2_vessel_detections_{start_date}.parquet


# Check data 